# Time Series Analyse: Chess Games über Zeit

Einfache Analyse von Trends und Mustern der Schach-Spiele über die Zeit

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Lade Daten und konvertiere Zeit-Spalten
df = pd.read_csv('games.csv')
df['created_at'] = pd.to_datetime(df['created_at'], unit='ms')
df = df.sort_values('created_at')

print(f"Datensatz: {len(df)} Spiele")
print(f"Zeitraum: {df['created_at'].min().date()} bis {df['created_at'].max().date()}")
print(f"Spieldauer: {(df['created_at'].max() - df['created_at'].min()).days} Tage")

In [ ]:
# Aggregiere nach Tag
df['avg_elo'] = (df['white_rating'] + df['black_rating']) / 2

daily_stats = df.set_index('created_at').resample('D').agg({
    'id': 'count',  # Anzahl Spiele pro Tag
    'avg_elo': 'mean',
    'turns': 'mean',
    'rated': 'sum'
}).rename(columns={'id': 'game_count', 'rated': 'rated_games'})

# Bereinige: Lösche Tage ohne Spiele
daily_stats = daily_stats[daily_stats['game_count'] > 0]

print(f"\nTägliche Statistiken:")
print(f"  Tage mit Spielen: {len(daily_stats)}")
print(f"  Durchschn. Spiele pro Tag: {daily_stats['game_count'].mean():.1f}")
print(f"  Min/Max Spiele pro Tag: {daily_stats['game_count'].min():.0f} / {daily_stats['game_count'].max():.0f}")
print(f"\nFirst 5 Tage:")
print(daily_stats.head())

In [ ]:
# Time Series Plots
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Time Series: Spiele über Zeit', fontsize=14, fontweight='bold')

# 1. Anzahl Spiele pro Tag
axes[0].plot(daily_stats.index, daily_stats['game_count'], marker='.', linestyle='-', linewidth=1, color='steelblue')
axes[0].fill_between(daily_stats.index, daily_stats['game_count'], alpha=0.3, color='steelblue')
axes[0].set_ylabel('Anzahl Spiele', fontsize=11)
axes[0].set_title('Spiele pro Tag')
axes[0].grid(alpha=0.3)

# 2. Durchschnittliche Elo-Rating
axes[1].plot(daily_stats.index, daily_stats['avg_elo'], linestyle='-', linewidth=1.5, color='green', alpha=0.7)
axes[1].set_ylabel('Average Elo', fontsize=11)
axes[1].set_xlabel('Zeit', fontsize=11)
axes[1].set_title('Durchschnittliche Spieler-Stärke (Average Elo)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Moving Averages
daily_stats['games_ma7'] = daily_stats['game_count'].rolling(window=7, center=True).mean()
daily_stats['games_ma30'] = daily_stats['game_count'].rolling(window=30, center=True).mean()
daily_stats['elo_ma7'] = daily_stats['avg_elo'].rolling(window=7, center=True).mean()

# Visualisiere mit Moving Averages
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('Time Series mit Gleitenden Durchschnitten (Moving Averages)', fontsize=14, fontweight='bold')

# 1. Spiele mit MA7 und MA30
axes[0].plot(daily_stats.index, daily_stats['game_count'], marker='.', linestyle='-', linewidth=0.5, 
             color='lightgray', label='Täglich', alpha=0.5)
axes[0].plot(daily_stats.index, daily_stats['games_ma7'], linewidth=2, color='orange', label='7-Tage MA')
axes[0].plot(daily_stats.index, daily_stats['games_ma30'], linewidth=2, color='red', label='30-Tage MA')
axes[0].set_ylabel('Anzahl Spiele', fontsize=11)
axes[0].set_title('Spiele pro Tag + Moving Averages')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Average Elo Rating mit MA7
axes[1].plot(daily_stats.index, daily_stats['avg_elo'], marker='.', linestyle='-', linewidth=0.5, 
             color='lightgray', label='Täglich', alpha=0.5)
axes[1].plot(daily_stats.index, daily_stats['elo_ma7'], linewidth=2, color='green', label='7-Tage MA')
axes[1].set_ylabel('Average Elo', fontsize=11)
axes[1].set_xlabel('Zeit', fontsize=11)
axes[1].set_title('Average Elo + 7-Tage Moving Average')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Trend-Analyse (einfache Regression)
from sklearn.linear_model import LinearRegression

# Vorbereitung
daily_stats_clean = daily_stats.dropna()
X = np.arange(len(daily_stats_clean)).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Trend-Analyse (Lineare Regression)', fontsize=14, fontweight='bold')

metrics = [
    ('game_count', axes[0], 'Spiele pro Tag'),
    ('avg_elo', axes[1], 'Average Elo')
]

trends = {}

for metric, ax, title in metrics:
    y = daily_stats_clean[metric].values
    
    # Fit Modell
    model = LinearRegression()
    model.fit(X, y)
    trend = model.predict(X)
    
    # Visualisiere
    ax.scatter(daily_stats_clean.index, y, alpha=0.3, s=10, color='gray', label='Täglich')
    ax.plot(daily_stats_clean.index, trend, linewidth=2, color='red', label='Trend')
    ax.set_title(title)
    ax.set_xlabel('Zeit')
    ax.set_ylabel(metric)
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Speichere Trend Info
    slope = model.coef_[0]
    r2 = model.score(X, y)
    trends[metric] = {'slope': slope, 'r2': r2}
    
    # Richtung
    direction = "📈 STEIGEND" if slope > 0 else "📉 FALLEND" if slope < 0 else "➡️  STABIL"
    ax.text(0.05, 0.95, f"{direction}\nSlope: {slope:.4f}\nR²: {r2:.3f}", 
            transform=ax.transAxes, verticalalignment='top', 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Zusammenfassung Trends
print("\n" + "="*70)
print("TREND-ANALYSE")
print("="*70)
for metric, values in trends.items():
    print(f"\n{metric}:")
    print(f"  Steigung (Slope): {values['slope']:.6f}")
    print(f"  R²-Wert: {values['r2']:.4f}")
    if values['slope'] > 0.001:
        print(f"  → STEIGENDER Trend (positiv)")
    elif values['slope'] < -0.001:
        print(f"  → FALLENDER Trend (negativ)")
    else:
        print(f"  → STABILER Trend")

In [ ]:
# Saisonale Muster: Nach Wochentag
df['day_of_week'] = df['created_at'].dt.day_name()
df['hour'] = df['created_at'].dt.hour

# Statistiken nach Wochentag
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_day_stats = df.groupby('day_of_week').agg({
    'id': 'count',
    'avg_elo': 'mean',
    'turns': 'mean'
}).reindex(day_order).rename(columns={'id': 'game_count'})

# Visualisierung
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Saisonale Muster: Nach Wochentag', fontsize=14, fontweight='bold')

# Spiele nach Wochentag
colors_week = ['orange' if day in ['Saturday', 'Sunday'] else 'steelblue' for day in day_order]
axes[0].bar(range(len(df_day_stats)), df_day_stats['game_count'], color=colors_week, alpha=0.7, edgecolor='black')
axes[0].set_xticks(range(len(df_day_stats)))
axes[0].set_xticklabels([d[:3] for d in day_order], rotation=45)
axes[0].set_ylabel('Anzahl Spiele')
axes[0].set_title('Spiele nach Wochentag')
axes[0].grid(alpha=0.3, axis='y')

# Average Elo nach Wochentag
axes[1].plot(range(len(df_day_stats)), df_day_stats['avg_elo'], marker='o', linewidth=2, markersize=8, color='green')
axes[1].set_xticks(range(len(df_day_stats)))
axes[1].set_xticklabels([d[:3] for d in day_order], rotation=45)
axes[1].set_ylabel('Average Elo')
axes[1].set_title('Average Elo nach Wochentag')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSpiele nach Wochentag:")
print(df_day_stats)

In [ ]:
# Zusammenfassung
print("\n" + "="*70)
print("ZUSAMMENFASSUNG: TIME SERIES ANALYSE")
print("="*70)

print(f"\n📅 ZEITRAUM")
print(f"   Von: {df['created_at'].min().date()}")
print(f"   Bis: {df['created_at'].max().date()}")
print(f"   Dauer: {(df['created_at'].max() - df['created_at'].min()).days} Tage")

print(f"\n📊 AKTIVITÄT")
print(f"   Gesamte Spiele: {len(df):,}")
print(f"   Durchschn. pro Tag: {daily_stats['game_count'].mean():.1f}")
print(f"   Min/Max pro Tag: {daily_stats['game_count'].min():.0f} / {daily_stats['game_count'].max():.0f}")

print(f"\n📈 TRENDS")
for metric, values in trends.items():
    if values['slope'] > 0.001:
        direction = "↗️ STEIGEND"
    elif values['slope'] < -0.001:
        direction = "↘️ FALLEND"
    else:
        direction = "→ STABIL"
    print(f"   {metric}: {direction} (Slope: {values['slope']:.4f})")

print(f"\n📅 SAISONALE MUSTER")
print(f"   Aktivster Tag: {df_day_stats['game_count'].idxmax()} ({df_day_stats['game_count'].max():.0f} Spiele)")
print(f"   Ruhigster Tag: {df_day_stats['game_count'].idxmin()} ({df_day_stats['game_count'].min():.0f} Spiele)")

# Wochenende vs Werkwoche
weekday_games = df_day_stats.loc[['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'], 'game_count'].mean()
weekend_games = df_day_stats.loc[['Saturday', 'Sunday'], 'game_count'].mean()
print(f"   Durchschn. Werkwoche: {weekday_games:.0f} Spiele")
print(f"   Durchschn. Wochenende: {weekend_games:.0f} Spiele")

if weekend_games > weekday_games:
    print(f"   → Mehr Aktivität am Wochenende (+{(weekend_games/weekday_games - 1)*100:.1f}%)")
else:
    print(f"   → Mehr Aktivität unter der Woche (+{(weekday_games/weekend_games - 1)*100:.1f}%)")